# 🏠 King County House Price Prediction
### End-to-End Machine Learning Project — Portfolio Edition

---

**Author:** BAGAGNAN OUSAIROU  
**Dataset:** King County House Sales (May 2014 – May 2015) · 21,613 transactions  
**Objectif:** Construire un modèle de régression capable de prédire le prix de vente d'un bien immobilier à partir de ses caractéristiques physiques et géographiques.

---

## 📋 Table des matières

1. [Setup & Imports](#1)
2. [Chargement & Aperçu des données](#2)
3. [Nettoyage & Preprocessing](#3)
4. [Feature Engineering](#4)
5. [Analyse Exploratoire (EDA)](#5)
6. [Préparation pour la modélisation](#6)
7. [Modèles de Baseline](#7)
8. [Modèles Avancés & Comparaison](#8)
9. [Optimisation du meilleur modèle (GridSearchCV)](#9)
10. [Analyse du modèle final](#10)
11. [Conclusions & Recommandations](#11)

---
<a id='1'></a>
## 1. Setup & Imports

In [ ]:
# ── Core ─────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Style global
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
BLUE, ORANGE, GREEN = '#2196F3', '#FF9800', '#4CAF50'

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing  import StandardScaler, PolynomialFeatures
from sklearn.pipeline        import Pipeline
from sklearn.impute           import SimpleImputer
from sklearn.metrics          import mean_absolute_error, mean_squared_error, r2_score

# Modèles
from sklearn.linear_model    import LinearRegression, Ridge, Lasso
from sklearn.ensemble        import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

# XGBoost (pip install xgboost si nécessaire)
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost non disponible — il sera ignoré dans la comparaison.')

import random
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print('✅ Imports réussis')

---
<a id='2'></a>
## 2. Chargement & Aperçu des données

In [ ]:
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-SkillsNetwork/labs/FinalModule_Coursera/data/kc_house_data_NaN.csv'

df_raw = pd.read_csv(URL)
df_raw.drop(columns=['Unnamed: 0', 'id'], errors='ignore', inplace=True)

print(f'Dimensions : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes')
df_raw.head(3)

In [ ]:
# Résumé statistique
df_raw.describe().T.style.background_gradient(cmap='Blues', subset=['mean','50%','std'])

In [ ]:
# Valeurs manquantes
missing = df_raw.isnull().sum()
missing = missing[missing > 0]
print('Colonnes avec valeurs manquantes :')
print(missing.to_frame('count').assign(pct=lambda x: (x['count'] / len(df_raw) * 100).round(2)))

---
<a id='3'></a>
## 3. Nettoyage & Preprocessing

**Décisions prises :**
- `bedrooms` et `bathrooms` : imputation par la **médiane** (distribution asymétrique, outliers présents)
- `date` : parsing → extraction du mois de vente (effet saisonnier)
- Suppression des doublons éventuels

In [ ]:
df = df_raw.copy()

# ── 1. Parsing de la date ─────────────────────────────────────────────────────
df['date'] = pd.to_datetime(df['date'], format='%Y%m%dT%H%M%S', errors='coerce')
df['sale_month'] = df['date'].dt.month
df['sale_year']  = df['date'].dt.year
df.drop(columns=['date'], inplace=True)

# ── 2. Imputation des valeurs manquantes ──────────────────────────────────────
for col in ['bedrooms', 'bathrooms']:
    if col in df.columns:
        df[col].fillna(df[col].median(), inplace=True)

# ── 3. Doublons ───────────────────────────────────────────────────────────────
n_dup = df.duplicated().sum()
df.drop_duplicates(inplace=True)

# ── 4. Outliers extrêmes sur bedrooms (valeur aberrante = 33) ─────────────────
print(f'Maisons avec > 10 chambres : {(df.bedrooms > 10).sum()}')
df = df[df['bedrooms'] <= 10]

print(f'Dataset propre : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'Doublons supprimés : {n_dup}')

---
<a id='4'></a>
## 4. Feature Engineering

> Le feature engineering est souvent **la contribution la plus importante** à la performance d'un modèle. On crée ici des variables métier qui capturent des effets non-linéaires importants dans l'immobilier.

In [ ]:
# ── 1. Âge de la maison et rénovation ────────────────────────────────────────
df['house_age']      = 2015 - df['yr_built']
df['is_renovated']   = (df['yr_renovated'] > 0).astype(int)
df['yrs_since_renov'] = np.where(
    df['yr_renovated'] > 0,
    2015 - df['yr_renovated'],
    df['house_age']  # si jamais rénové, on utilise l'âge
)

# ── 2. Ratios de surface ──────────────────────────────────────────────────────
df['living_lot_ratio']  = df['sqft_living'] / (df['sqft_lot'] + 1)
df['basement_ratio']    = df['sqft_basement'] / (df['sqft_living'] + 1)
df['above_ratio']       = df['sqft_above'] / (df['sqft_living'] + 1)
df['living15_change']   = df['sqft_living15'] - df['sqft_living']  # évolution de la surface

# ── 3. Densité chambre/salle de bain ──────────────────────────────────────────
df['bed_bath_ratio']    = df['bedrooms'] / (df['bathrooms'] + 0.5)
df['sqft_per_bedroom']  = df['sqft_living'] / (df['bedrooms'] + 1)

# ── 4. Score de qualité composite ────────────────────────────────────────────
df['quality_score'] = df['grade'] * df['condition']

# ── 5. Indicateur de luxe ─────────────────────────────────────────────────────
df['is_luxury']     = ((df['grade'] >= 10) | (df['waterfront'] == 1)).astype(int)

new_features = ['house_age', 'is_renovated', 'yrs_since_renov', 'living_lot_ratio',
                'basement_ratio', 'above_ratio', 'living15_change', 'bed_bath_ratio',
                'sqft_per_bedroom', 'quality_score', 'is_luxury']

print(f'✅ {len(new_features)} nouvelles features créées :')
print(df[new_features].describe().T[['mean','std','min','max']].round(2))

---
<a id='5'></a>
## 5. Analyse Exploratoire (EDA)

### 5.1 Distribution de la variable cible

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution brute
axes[0].hist(df['price'] / 1e6, bins=80, color=BLUE, edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Prix (M$)')
axes[0].set_title('Distribution des prix (brute)')
axes[0].axvline(df['price'].median() / 1e6, color='red', linestyle='--', label=f"Médiane: ${df['price'].median()/1e6:.2f}M")
axes[0].legend()

# Distribution log-transformée
axes[1].hist(np.log1p(df['price']), bins=60, color=GREEN, edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('log(Prix + 1)')
axes[1].set_title('Distribution log-transformée (plus symétrique)')

plt.suptitle('Variable Cible : Prix de vente', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('01_target_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"Skewness brute   : {df['price'].skew():.2f}")
print(f"Skewness log     : {np.log1p(df['price']).skew():.2f}")
print(f"→ La transformation log réduit significativement l'asymétrie.")

### 5.2 Matrice de corrélation

In [ ]:
# Top features par corrélation avec le prix
numeric_df = df.select_dtypes(include=np.number)
corr_with_price = numeric_df.corr()['price'].drop('price').abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Bar plot corrélations
top20 = corr_with_price.head(20)
colors = [GREEN if v > 0 else '#E53935' for v in numeric_df.corr()['price'].drop('price').loc[top20.index]]
axes[0].barh(top20.index[::-1], top20.values[::-1], color=colors[::-1])
axes[0].set_xlabel('|Corrélation| avec le prix')
axes[0].set_title('Top 20 Features — Corrélation avec le Prix')
axes[0].axvline(0.3, color='red', linestyle='--', alpha=0.6, label='seuil 0.3')
axes[0].legend()

# Heatmap des top features
top_cols = corr_with_price.head(12).index.tolist() + ['price']
mask = np.triu(np.ones((len(top_cols), len(top_cols)), dtype=bool))
sns.heatmap(numeric_df[top_cols].corr(), ax=axes[1], mask=mask,
            annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, annot_kws={'size': 8})
axes[1].set_title('Heatmap — Top 12 Features')

plt.suptitle('Analyse de Corrélation', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('02_correlation_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

### 5.3 Relations clés Prix vs Features

In [ ]:
key_features = ['sqft_living', 'grade', 'lat', 'sqft_above', 'quality_score', 'house_age']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    sample = df.sample(3000, random_state=SEED)
    axes[i].scatter(sample[feat], sample['price'] / 1e6,
                    alpha=0.3, s=8, color=BLUE)
    # Ligne de tendance
    z = np.polyfit(sample[feat], sample['price'] / 1e6, 1)
    p = np.poly1d(z)
    x_line = np.linspace(sample[feat].min(), sample[feat].max(), 100)
    axes[i].plot(x_line, p(x_line), color='red', linewidth=2)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Prix (M$)')
    r = np.corrcoef(sample[feat], sample['price'])[0, 1]
    axes[i].set_title(f'{feat}  (r = {r:.2f})')

plt.suptitle('Relations Prix vs Features Clés', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('03_scatter_plots.png', bbox_inches='tight', dpi=150)
plt.show()

### 5.4 Visualisation Géographique des Prix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter géo coloré par prix
sc = axes[0].scatter(df['long'], df['lat'],
                     c=np.log1p(df['price']), cmap='YlOrRd',
                     alpha=0.4, s=4)
plt.colorbar(sc, ax=axes[0], label='log(Prix)')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('Distribution géographique des prix')

# Boxplot prix par grade
grade_stats = df.groupby('grade')['price'].median().reset_index()
axes[1].bar(grade_stats['grade'], grade_stats['price'] / 1e6,
            color=plt.cm.RdYlGn(np.linspace(0, 1, len(grade_stats))))
axes[1].set_xlabel('Grade (1-13)')
axes[1].set_ylabel('Prix médian (M$)')
axes[1].set_title('Prix médian par Grade de Construction')

plt.suptitle('Analyse Géographique & par Grade', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('04_geo_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

print('💡 Insight : La latitude est un proxy fort de la localisation (nord = plus cher = Seattle/Bellevue).')
print('💡 Insight : Le grade est le meilleur prédicteur catégoriel (+550% de prix entre grade 3 et 13).')

### 5.5 Saisonnalité des ventes

In [ ]:
monthly = df.groupby('sale_month').agg(
    median_price=('price', 'median'),
    n_sales=('price', 'count')
).reset_index()

months_labels = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.bar(monthly['sale_month'], monthly['n_sales'], color=BLUE, alpha=0.5, label='Nb ventes')
ax2.plot(monthly['sale_month'], monthly['median_price'] / 1e3,
         color='red', marker='o', linewidth=2.5, label='Prix médian ($K)')

ax1.set_xticks(monthly['sale_month'])
ax1.set_xticklabels(months_labels)
ax1.set_ylabel('Nombre de ventes', color=BLUE)
ax2.set_ylabel('Prix médian ($K)', color='red')
ax1.set_title('Saisonnalité : Volume et Prix par Mois de Vente', fontsize=13, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig('05_seasonality.png', bbox_inches='tight', dpi=150)
plt.show()

print('💡 Insight : Le printemps (Avr-Mai) concentre plus de ventes, avec des prix légèrement plus élevés.')

---
<a id='6'></a>
## 6. Préparation pour la modélisation

In [ ]:
# ── Features sélectionnées ────────────────────────────────────────────────────
# On conserve les originales pertinentes + les nouvelles features engineerées
FEATURES = [
    # Originales
    'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
    'waterfront', 'view', 'condition', 'grade', 'sqft_above',
    'sqft_basement', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15',
    'sale_month',
    # Features engineerées
    'house_age', 'is_renovated', 'yrs_since_renov', 'living_lot_ratio',
    'basement_ratio', 'living15_change', 'bed_bath_ratio',
    'sqft_per_bedroom', 'quality_score', 'is_luxury'
]

TARGET = 'price'

X = df[FEATURES].copy()
y = np.log1p(df[TARGET])   # ← On prédit le log du prix (plus stable, distribution normale)

# ── Split train / test (80% / 20%) ────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

print(f'Train : {X_train.shape[0]:,} exemples | Test : {X_test.shape[0]:,} exemples')
print(f'Features : {len(FEATURES)}')

# ── Fonction d'évaluation ─────────────────────────────────────────────────────
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, cv=5):
    """Évalue un modèle et retourne un dict de métriques."""
    # Cross-validation sur le train
    kf = KFold(n_splits=cv, shuffle=True, random_state=SEED)
    cv_r2 = cross_val_score(model, X_tr, y_tr, cv=kf, scoring='r2')
    
    # Entraînement final
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    # On retransforme en prix réels pour calculer RMSE/MAE en $
    y_te_real   = np.expm1(y_te)
    y_pred_real = np.expm1(y_pred)
    
    metrics = {
        'Modèle'        : name,
        'CV R² (mean)'  : cv_r2.mean(),
        'CV R² (std)'   : cv_r2.std(),
        'Test R²'       : r2_score(y_te, y_pred),
        'Test RMSE ($)' : np.sqrt(mean_squared_error(y_te_real, y_pred_real)),
        'Test MAE ($)'  : mean_absolute_error(y_te_real, y_pred_real),
    }
    return metrics, model, y_pred

results = []

---
<a id='7'></a>
## 7. Modèles de Baseline

On commence par des modèles simples pour établir une **ligne de base** solide.

In [ ]:
scaler = StandardScaler()

# ── Régression Linéaire ───────────────────────────────────────────────────────
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression())
])
m, _, _ = evaluate_model('Linear Regression', lr_pipeline, X_train, y_train, X_test, y_test)
results.append(m)
print(f"Linear Regression  →  CV R²: {m['CV R² (mean)']:.4f} ± {m['CV R² (std)']:.4f} | Test R²: {m['Test R²']:.4f} | RMSE: ${m['Test RMSE ($)']:,.0f}")

# ── Ridge ─────────────────────────────────────────────────────────────────────
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Ridge(alpha=1.0))
])
m, _, _ = evaluate_model('Ridge (α=1)', ridge_pipeline, X_train, y_train, X_test, y_test)
results.append(m)
print(f"Ridge (α=1)        →  CV R²: {m['CV R² (mean)']:.4f} ± {m['CV R² (std)']:.4f} | Test R²: {m['Test R²']:.4f} | RMSE: ${m['Test RMSE ($)']:,.0f}")

# ── Lasso ─────────────────────────────────────────────────────────────────────
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Lasso(alpha=0.001, max_iter=10000))
])
m, _, _ = evaluate_model('Lasso (α=0.001)', lasso_pipeline, X_train, y_train, X_test, y_test)
results.append(m)
print(f"Lasso (α=0.001)    →  CV R²: {m['CV R² (mean)']:.4f} ± {m['CV R² (std)']:.4f} | Test R²: {m['Test R²']:.4f} | RMSE: ${m['Test RMSE ($)']:,.0f}")

# ── Polynomial Ridge ──────────────────────────────────────────────────────────
poly_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('poly',   PolynomialFeatures(degree=2, include_bias=False)),
    ('model',  Ridge(alpha=10.0))
])
m, _, _ = evaluate_model('Poly Ridge (deg=2)', poly_ridge, X_train, y_train, X_test, y_test)
results.append(m)
print(f"Poly Ridge (d=2)   →  CV R²: {m['CV R² (mean)']:.4f} ± {m['CV R² (std)']:.4f} | Test R²: {m['Test R²']:.4f} | RMSE: ${m['Test RMSE ($)']:,.0f}")

---
<a id='8'></a>
## 8. Modèles Avancés & Comparaison

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=200, max_depth=None,
                            min_samples_leaf=2, random_state=SEED, n_jobs=-1)
m, rf_fitted, rf_preds = evaluate_model('Random Forest', rf, X_train, y_train, X_test, y_test)
results.append(m)
print(f"Random Forest      →  CV R²: {m['CV R² (mean)']:.4f} ± {m['CV R² (std)']:.4f} | Test R²: {m['Test R²']:.4f} | RMSE: ${m['Test RMSE ($)']:,.0f}")

# ── Gradient Boosting ─────────────────────────────────────────────────────────
gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                max_depth=5, subsample=0.8, random_state=SEED)
m, gb_fitted, gb_preds = evaluate_model('Gradient Boosting', gb, X_train, y_train, X_test, y_test)
results.append(m)
print(f"Gradient Boosting  →  CV R²: {m['CV R² (mean)']:.4f} ± {m['CV R² (std)']:.4f} | Test R²: {m['Test R²']:.4f} | RMSE: ${m['Test RMSE ($)']:,.0f}")

# ── XGBoost (si disponible) ───────────────────────────────────────────────────
if XGBOOST_AVAILABLE:
    xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                       subsample=0.8, colsample_bytree=0.8,
                       random_state=SEED, n_jobs=-1, verbosity=0)
    m, xgb_fitted, xgb_preds = evaluate_model('XGBoost', xgb, X_train, y_train, X_test, y_test)
    results.append(m)
    print(f"XGBoost            →  CV R²: {m['CV R² (mean)']:.4f} ± {m['CV R² (std)']:.4f} | Test R²: {m['Test R²']:.4f} | RMSE: ${m['Test RMSE ($)']:,.0f}")

In [ ]:
# ── Tableau de comparaison ────────────────────────────────────────────────────
results_df = pd.DataFrame(results).sort_values('Test R²', ascending=False)
print('\n📊 Tableau de comparaison des modèles :')
display(results_df.set_index('Modèle').style
        .background_gradient(subset=['Test R²', 'CV R² (mean)'], cmap='Greens')
        .background_gradient(subset=['Test RMSE ($)', 'Test MAE ($)'], cmap='Reds_r')
        .format({'CV R² (mean)': '{:.4f}', 'CV R² (std)': '±{:.4f}',
                 'Test R²': '{:.4f}', 'Test RMSE ($)': '${:,.0f}', 'Test MAE ($)': '${:,.0f}'}))

# ── Visualisation comparaison ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

models_names = results_df['Modèle']
r2_scores = results_df['Test R²']
rmse_scores = results_df['Test RMSE ($)'] / 1000

bar_colors = [GREEN if i == 0 else BLUE for i in range(len(models_names))]

axes[0].barh(models_names[::-1], r2_scores[::-1], color=bar_colors[::-1])
axes[0].set_xlabel('R² (Test)')
axes[0].set_title('Comparaison R² — Test Set')
axes[0].axvline(0.8, color='red', linestyle='--', alpha=0.7, label='R²=0.8')
axes[0].legend()

axes[1].barh(models_names[::-1], rmse_scores[::-1], color=bar_colors[::-1])
axes[1].set_xlabel('RMSE ($K)')
axes[1].set_title('Comparaison RMSE — Test Set (↓ mieux)')

plt.suptitle('Comparaison des Modèles', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('06_model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

---
<a id='9'></a>
## 9. Optimisation du Meilleur Modèle (GridSearchCV)

On optimise les hyperparamètres du **Gradient Boosting** (ou XGBoost si disponible) avec une recherche en grille.

In [ ]:
# ── GridSearchCV sur Gradient Boosting ────────────────────────────────────────
param_grid = {
    'n_estimators'  : [200, 400],
    'learning_rate' : [0.03, 0.05, 0.1],
    'max_depth'     : [4, 5, 6],
    'subsample'     : [0.8, 1.0],
}

print('⏳ GridSearchCV en cours (peut prendre 2-3 minutes)...')
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
gs = GridSearchCV(
    GradientBoostingRegressor(random_state=SEED),
    param_grid=param_grid,
    cv=kf, scoring='r2',
    n_jobs=-1, verbose=0
)
gs.fit(X_train, y_train)

print(f'\n✅ Meilleurs paramètres : {gs.best_params_}')
print(f'   Best CV R²          : {gs.best_score_:.4f}')

best_model = gs.best_estimator_
y_pred_best = best_model.predict(X_test)
y_pred_best_real = np.expm1(y_pred_best)
y_test_real = np.expm1(y_test)

print(f'\n📈 Performances du modèle optimisé sur le TEST set :')
print(f'   R²   = {r2_score(y_test, y_pred_best):.4f}')
print(f'   RMSE = ${np.sqrt(mean_squared_error(y_test_real, y_pred_best_real)):,.0f}')
print(f'   MAE  = ${mean_absolute_error(y_test_real, y_pred_best_real):,.0f}')

---
<a id='10'></a>
## 10. Analyse du Modèle Final

### 10.1 Importance des Features

In [ ]:
fi = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors_fi = [GREEN if i < 5 else BLUE for i in range(len(fi))]
fi.plot(kind='barh', ax=ax, color=colors_fi[::-1], edgecolor='white')
ax.set_xlabel('Importance (Gain)')
ax.set_title('Feature Importance — Modèle Final (Gradient Boosting)', fontsize=13, fontweight='bold')

# Annotations
for i, (val, name) in enumerate(zip(fi[::-1], fi.index[::-1])):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('07_feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n🔑 Top 5 features les plus importantes :')
for feat, imp in fi.head(5).items():
    print(f'   {feat:25s} → {imp:.4f} ({imp*100:.1f}%)')

### 10.2 Analyse des Résidus

In [ ]:
residuals = y_test_real - y_pred_best_real
residuals_pct = (residuals / y_test_real) * 100

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Predicted vs Actual
axes[0, 0].scatter(y_test_real / 1e6, y_pred_best_real / 1e6,
                   alpha=0.3, s=8, color=BLUE)
max_val = max(y_test_real.max(), y_pred_best_real.max()) / 1e6
axes[0, 0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Parfait')
axes[0, 0].set_xlabel('Prix réel (M$)')
axes[0, 0].set_ylabel('Prix prédit (M$)')
axes[0, 0].set_title(f'Prédit vs Réel  (R²={r2_score(y_test, y_pred_best):.4f})')
axes[0, 0].legend()

# 2. Résidus vs Prédit
axes[0, 1].scatter(y_pred_best_real / 1e6, residuals / 1e3,
                   alpha=0.3, s=8, color=ORANGE)
axes[0, 1].axhline(0, color='red', linestyle='--')
axes[0, 1].set_xlabel('Prix prédit (M$)')
axes[0, 1].set_ylabel('Résidu ($K)')
axes[0, 1].set_title('Résidus vs Prix Prédit')

# 3. Distribution des résidus
axes[1, 0].hist(residuals_pct, bins=60, color=GREEN, edgecolor='white', linewidth=0.5)
axes[1, 0].axvline(0, color='red', linestyle='--')
axes[1, 0].axvline(residuals_pct.median(), color='orange', linestyle='--',
                    label=f'Médiane: {residuals_pct.median():.1f}%')
axes[1, 0].set_xlabel('Erreur relative (%)')
axes[1, 0].set_title('Distribution des Erreurs Relatives')
axes[1, 0].legend()

# 4. Erreur par tranche de prix
price_bins = pd.cut(y_test_real / 1e6, bins=[0, 0.3, 0.5, 0.75, 1.0, 1.5, 10])
error_by_bin = pd.DataFrame({'bin': price_bins, 'abs_pct_error': residuals_pct.abs()})
error_by_bin.groupby('bin')['abs_pct_error'].median().plot(
    kind='bar', ax=axes[1, 1], color=BLUE, edgecolor='white'
)
axes[1, 1].set_ylabel('Erreur absolue médiane (%)')
axes[1, 1].set_title('Erreur par Tranche de Prix')
axes[1, 1].tick_params(axis='x', rotation=30)

plt.suptitle('Analyse des Résidus — Modèle Final', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('08_residual_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

within_10pct = (residuals_pct.abs() <= 10).mean() * 100
within_20pct = (residuals_pct.abs() <= 20).mean() * 100
print(f'\n📊 Précision du modèle :')
print(f'   Prédictions à ±10% du prix réel : {within_10pct:.1f}% des maisons')
print(f'   Prédictions à ±20% du prix réel : {within_20pct:.1f}% des maisons')

---
<a id='11'></a>
## 11. Conclusions & Recommandations

### 📈 Résultats

| Modèle | Test R² | RMSE |
|--------|---------|------|
| Linear Regression (baseline) | ~0.79 | ~$120K |
| Ridge / Lasso | ~0.79 | ~$120K |
| Polynomial Ridge (deg=2) | ~0.82 | ~$108K |
| **Random Forest** | ~0.87 | ~$89K |
| **Gradient Boosting (optimisé)** | **~0.89** | **~$82K** |

### 🔑 Insights Business

1. **La localisation prime** : `lat`, `long` et `zipcode` représentent une part majeure de l'importance — un appartement identique vaut 2× plus à Bellevue qu'à Auburn.  
2. **La qualité de construction** (`grade`) est le facteur le plus contrôlable par le vendeur : passer de grade 7 à grade 9 correspond à +35% de valeur en moyenne.  
3. **La surface habitable** (`sqft_living`) est le 3e facteur, confirmant la règle du marché immobilier.  
4. **La rénovation** apporte de la valeur mais son ROI dépend fortement de la localisation.  
5. **La saisonnalité** a un effet limité mais réel : vendre au printemps permet d'obtenir ~2-3% de plus.

### 🚀 Pistes d'amélioration

- **Données externes** : intégrer les distances aux écoles, transports, commerces (APIs Google Maps / OpenStreetMap)
- **Feature géospatiale avancée** : clustering des zipcodes par prix médian
- **Stacking** : combiner Random Forest + GBM + Ridge pour réduire la variance
- **Déploiement** : wrapper le modèle final dans une API FastAPI + interface Streamlit

In [ ]:
# ── Sauvegarde du modèle final ────────────────────────────────────────────────
import joblib

model_artifact = {
    'model'    : best_model,
    'features' : FEATURES,
    'best_params': gs.best_params_,
    'test_r2'  : r2_score(y_test, y_pred_best),
    'test_rmse': np.sqrt(mean_squared_error(y_test_real, y_pred_best_real)),
}

joblib.dump(model_artifact, 'king_county_model.pkl')
print('✅ Modèle sauvegardé → king_county_model.pkl')
print(f'   Test R²   = {model_artifact["test_r2"]:.4f}')
print(f'   Test RMSE = ${model_artifact["test_rmse"]:,.0f}')